In [10]:
# Deleting old vector stores if present

import shutil
from pathlib import Path

for folder in ["./chroma_db", "./faiss_db"]:
    if Path(folder).exists():
        shutil.rmtree(folder)
        print(f"Deleted {folder}")
    else:
        print(f"No {folder} found")

No ./chroma_db found
Deleted ./faiss_db


In [11]:
# Importing all required packages

import os
import fitz
from pathlib import Path
from dotenv import load_dotenv
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

load_dotenv()
print("All imports successful!")

All imports successful!


In [12]:
# Loading PDFs from ncert_chapters folder

PDF_FOLDER = "ncert_chapters"

all_docs = []
for pdf_path in Path(PDF_FOLDER).glob("*.pdf"):
    pdf = fitz.open(str(pdf_path))
    for page_num in range(len(pdf)):
        page = pdf[page_num]
        text = page.get_text()
        if text.strip():
            all_docs.append(Document(
                page_content=text,
                metadata={"source": pdf_path.name, "page": page_num + 1}
            ))
    pdf.close()
    print(f"Loaded: {pdf_path.name}")

print(f"\nTotal pages loaded: {len(all_docs)}")

Loaded: chapter1.pdf
Loaded: chapter10.pdf
Loaded: chapter11.pdf
Loaded: chapter12.pdf
Loaded: chapter13.pdf
Loaded: chapter2.pdf
Loaded: chapter3.pdf
Loaded: chapter4.pdf
Loaded: chapter5.pdf
Loaded: chapter6.pdf
Loaded: chapter7.pdf
Loaded: chapter8.pdf
Loaded: chapter9.pdf

Total pages loaded: 217


In [13]:
# Splitting PDF text into chunks

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.split_documents(all_docs)
print(f"Total chunks: {len(chunks)}")

Total chunks: 626


In [14]:
# Embedding chunks and creating FAISS vector store

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

# Save FAISS index locally
vectorstore.save_local("./faiss_db")
print("FAISS vector store created and saved to ./faiss_db!")

FAISS vector store created and saved to ./faiss_db!


In [15]:
# Loading FAISS and setting up LLM

api_key = os.environ.get("SARVAM_API_KEY")
print(f"API key loaded: {'Yes' if api_key else 'NO - check your .env file'}")

llm = ChatOpenAI(
    base_url="https://api.sarvam.ai/v1",
    api_key=api_key,
    model="sarvam-m",
    temperature=0.3,
    max_tokens=200
)
print("LLM ready!")

API key loaded: Yes
LLM ready!


In [16]:
# Creating RAG chain

# Load FAISS retriever
faiss_store = FAISS.load_local(
    "./faiss_db",
    embedding_model,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 7}
)

# Prompt template
prompt_template = """You are a strict but caring CBSE Science mentor for Class 10 students.
Your personality: direct, clear, no-nonsense — like a good school teacher who genuinely wants students to learn.
Do NOT be overly friendly or use excessive praise. Be warm but professional.

GREETING RULE — VERY IMPORTANT:
- If the student says hi, hello, hey, thanks, thank you, bye, or any casual greeting or farewell:
  Respond naturally and briefly like a teacher would. Do NOT mention any science topic unprompted.
  Example responses: "Hello! What would you like to study today?" or "You're welcome. Let me know if you have more questions."

ANSWER LENGTH RULES — follow these strictly:
- Simple factual or definition questions (e.g. "What is chlorophyll?"): Answer in 2-3 sentences. Highlight the key term.
- Conceptual or process-based questions (e.g. "Explain photosynthesis"): Answer in detail with steps or structure. Use bullet points if helpful.
- Numerical or formula questions: Show the formula, then solve step by step.

OUT-OF-SYLLABUS RULE:
- If the answer IS in the context below: answer from it directly.
- If the answer is NOT in the context: Start with "[Outside NCERT Syllabus]" then answer briefly from general scientific knowledge. Never make up facts.

NEVER say "I don't know" — either use the textbook context or answer from general knowledge with the label above.

Context:
{context}

Student's Question: {question}

Answer (short and simple but use bullet points when needed):"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=True
)
print("RAG chain ready!")

RAG chain ready!


In [17]:
# Test query

question = "Human brain explanation"
result = qa_chain.invoke({"query": question})

raw_answer = result["result"]

# Clean <think> tags if present
if "</think>" in raw_answer:
    final_answer = raw_answer.split("</think>")[-1].strip()
else:
    final_answer = raw_answer.strip()

print("ANSWER:\n")
print(final_answer)

print("\n--- Sources ---")
for doc in result["source_documents"]:
    print(f"Page {doc.metadata.get('page')} | {doc.metadata.get('source')}")

ANSWER:

Here's a concise explanation of the human brain based on your syllabus context:

**Human Brain Structure & Functions:**  
- **Fore-brain (Cerebrum):**  
  - Main thinking center for decision-making  
  - Processes sensory inputs (sight, sound, smell)  
  - Contains association areas to interpret information  

- **Mid-brain:**  
  - Coordinates involuntary actions (e.g., heartbeats, pupil reflexes)  
  - Relays visual/auditory signals  

- **Hind-brain (Cerebellum & Medulla):**  
  - **Cerebellum:** Controls voluntary movements (walking, balance), posture  
  - **Medulla:** Manages vital involuntary functions (breathing, blood pressure)  

**Key Features:**  
- Receives signals via peripheral nerves (spinal cord/cranial nerves)  
- Sends responses through motor neurons to muscles/glands  
- Integrates stored memory with real-time sensory data for actions  

For reflexes (e.g., hand withdrawal), the spinal cord handles immediate responses, but complex actions require brain proc